In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, confusion_matrix, f1_score

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/integrate_m3c/merged.processed.harmony.h5ad')

In [ ]:
adata

In [ ]:
adata.obs.head()

In [ ]:
adata.obs.index = [x[:-2] for x in adata.obs.index]

In [ ]:
sc.pl.umap(adata, color=['assay', 'time', 'line'])

In [ ]:
# indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/'
# clusters = pd.read_csv(f'{indir}/csv/label_transfer/snmct_subsample.v6.csv', sep='\t', header=0, index_col=0)
# clusters.rename(columns={'pred_cluster' : 'pred_coarse'}, inplace=True)
# clusters.shape

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/'
clusters = pd.read_csv(f'{indir}/csv/label_transfer/snmct_xgboost_v7.csv', sep='\t', header=0, index_col=0)
clusters.shape

In [ ]:
clusters.head()

In [ ]:
adata.obs['rna_cluster'] = clusters.reindex(adata.obs.index)['pred_coarse']
tmp_adata = adata[~adata.obs['rna_cluster'].isna()]

In [ ]:
tmp_adata

In [ ]:
snmct = tmp_adata[tmp_adata.obs['assay'] == 'snmCT']
X = snmct.obsm['X_pca_harmony'][:, :50]
y = snmct.obs['rna_cluster']
X.shape, y.shape

In [ ]:
### no time

features = np.hstack([snmct.obsm['X_pca_harmony']])

X = features
y = snmct.obs['rna_cluster']
train_size = 0.8
test_size = 0.2

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, train_size=train_size)
model = OneVsRestClassifier(XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
model.fit(X_train, y_train)
xgboost_pred = model.predict(X_test)
print(classification_report(y_test, xgboost_pred))

In [ ]:
### with time
features = np.hstack([snmct.obsm['X_pca_harmony'], snmct.obs['time'].values.reshape(-1, 1)])

X = features
y = snmct.obs['rna_cluster']
train_size = 0.8
test_size = 0.2

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, train_size=train_size)
model = OneVsRestClassifier(XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
model.fit(X_train, y_train)
xgboost_pred = model.predict(X_test)
print(classification_report(y_test, xgboost_pred))

In [ ]:
y_test.unique()

In [ ]:
tmp_order = ['Start', 'Sendai', 'Skin', 'Inflamed', 'Delayed', 'Fail1', 'Inter1', 'Inter2', 'Fail2','IPS']
conf_mtx = pd.DataFrame(confusion_matrix(y_test, xgboost_pred, labels=tmp_order))
conf_mtx = conf_mtx.div(conf_mtx.sum(axis=0), axis=1).fillna(0)
conf_mtx.columns = tmp_order
conf_mtx.index = tmp_order
conf_mtx

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv'
conf_mtx.to_csv(f'{outdir}/figure1/mct_xgboost_conf_mtx.csv', sep='\t')

In [ ]:
clusters = sorted(snmct.obs['rna_cluster'].unique())
n_classes = snmct.obs['rna_cluster'].unique().shape[0]
tmp_clust = dict(zip(list(range(n_classes)), clusters))
clust_tmp = dict(zip(clusters, list(range(n_classes)),))

In [ ]:
snmct.obs['tmp_cluster'] = [clust_tmp.get(x) for x in snmct.obs['rna_cluster']]

In [ ]:
# features = np.hstack([snmct.obsm['X_pca_harmony'], snmct.obs['time'].values.reshape(-1, 1)])

# X = features
# y = snmct.obs['tmp_cluster']
# train_size = 0.8
# test_size = 0.2

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, train_size=train_size)
# model = XGBClassifier(
#     objective='multi:softprob',          # native multiclass objective
#     num_class=n_classes,                 # total number of classes
#     use_label_encoder=False,
#     eval_metric='mlogloss',              # suitable for multiclass
#     random_state=42
# )

# model.fit(X_train, y_train)
# xgboost_pred = model.predict(X_test)
# print(classification_report(y_test, xgboost_pred))

### use the 50 harmony PCs + time on all of the data

In [ ]:
features = np.hstack([snmct.obsm['X_pca_harmony'], snmct.obs['time'].values.reshape(-1, 1)])

X = features
y = snmct.obs['rna_cluster']
model = OneVsRestClassifier(XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))

model.fit(X, y)

In [ ]:
snm3c = adata[adata.obs['assay'] == 'snm3C']
features = np.hstack([snm3c.obsm['X_pca_harmony'], snm3c.obs['time'].values.reshape(-1, 1)])
snm3c.obs['pred_cluster'] = model.predict(features)

In [ ]:
snm3c.obs['pred_cluster'].value_counts()

### check higashi really quickly

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/higashi'
higashi = sc.read_h5ad(f'{indir}/h5ad/embedding/higashi_filtered.h5ad')

In [ ]:
sc.pl.umap(higashi, color=['line', 'time'])

In [ ]:
higashi.obs.head()

In [ ]:
higashi.obs['cluster'] = snm3c.obs.reindex(higashi.obs.index)['pred_cluster']
tmp_higashi = higashi[~higashi.obs['cluster'].isna()]

In [ ]:
sc.pl.umap(tmp_higashi, color=['cluster'])

In [ ]:
# sc.pl.umap(tmp_higashi, color=['cluster'])

In [ ]:
def cluster_small_multiples(adata, clust_key, size=10, ncols=2,
                            frameon=False, legend_loc=None, **kwargs):
    tmp = adata.copy()

    for i,clust in enumerate(adata.obs[clust_key].cat.categories):
        tmp.obs[clust] = adata.obs[clust_key].isin([clust]).astype('category')
        tmp.uns[clust+'_colors'] = ['#d3d3d3', adata.uns[clust_key+'_colors'][i]]

    sc.pl.umap(tmp, groups=tmp.obs[clust].cat.categories[1:].values,
               color=adata.obs[clust_key].cat.categories.tolist(),
               size=size, frameon=frameon, legend_loc=legend_loc,
               ncols=ncols, **kwargs)

In [ ]:
cluster_small_multiples(tmp_higashi, 'cluster', size=50, ncols=4)

In [ ]:
snm3c.obs.head()

In [ ]:
sc.pl.umap(tmp_higashi, color=['cluster', 'leiden'])

In [ ]:
res = 1.0
sc.tl.leiden(tmp_higashi, resolution=res, random_state=0)

In [ ]:
sc.pl.umap(tmp_higashi, color=['cluster', 'leiden'])

In [ ]:
tmp_higashi.obs['tmp_leiden'] = [f'leiden{x}' for x in tmp_higashi.obs['leiden']]
tmp_higashi.obs['tmp_leiden'] = tmp_higashi.obs['tmp_leiden'].astype('category')

In [ ]:
# cluster_small_multiples(tmp_higashi, 'tmp_leiden', size=50, ncols=3)

In [ ]:
tmp_higashi.obs['cluster'].value_counts(), \
snm3c.obs['pred_cluster'].value_counts()

In [ ]:
tmp_higashi.obs['cluster_sub'] = np.nan

###
clust = 'Inter1'
whitelist = ['1', '3']
mask = [x in whitelist and y == f'{clust}' for x,y in zip(tmp_higashi.obs['leiden'], tmp_higashi.obs['cluster'])]
tmp_higashi.obs.loc[mask, 'cluster_sub'] = f'{clust}_1'
mask = [x != f'{clust}_1' and y == f'{clust}' for x,y in zip(tmp_higashi.obs['cluster_sub'], tmp_higashi.obs['cluster'])]
tmp_higashi.obs.loc[mask, 'cluster_sub'] = f'{clust}_2'

###
clust = 'Delayed'
whitelist = ['1', '3']
mask = [x in whitelist and y == f'{clust}' for x,y in zip(tmp_higashi.obs['leiden'], tmp_higashi.obs['cluster'])]
tmp_higashi.obs.loc[mask, 'cluster_sub'] = f'{clust}_1'
mask = [x != f'{clust}_1' and y == f'{clust}' for x,y in zip(tmp_higashi.obs['cluster_sub'], tmp_higashi.obs['cluster'])]
tmp_higashi.obs.loc[mask, 'cluster_sub'] = f'{clust}_2'

###
clust = 'IPS'
whitelist = ['6']
mask = [x in whitelist and y == f'{clust}' for x,y in zip(tmp_higashi.obs['leiden'], tmp_higashi.obs['cluster'])]
tmp_higashi.obs.loc[mask, 'cluster_sub'] = f'{clust}_1'

mask = [x != f'{clust}_1' and y == f'{clust}' for x,y in zip(tmp_higashi.obs['cluster_sub'], tmp_higashi.obs['cluster'])]
tmp_higashi.obs.loc[mask, 'cluster_sub'] = f'{clust}_2'

mask = tmp_higashi.obs['cluster_sub'].isna()
tmp_higashi.obs.loc[mask, 'cluster_sub'] = tmp_higashi.obs.loc[mask, 'cluster'].copy()

In [ ]:
tmp_higashi.obs['cluster_sub'].value_counts()

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/mc'
select = ['time', 'line', 'cluster', 'cluster_sub']
tmp_higashi.obs[select].to_csv(f'{outdir}/csv/label_transfer/xgboost_time_v7.csv', sep='\t')

In [ ]:
!date